In [12]:

from pathlib import Path

import pandas as pd

In [13]:
PROJECT_ROOT = Path.cwd().parents[1]
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "gles_mip"

v1_review_path = OUTPUT_DIR / "gles_mip_v1_review_template.csv"
v2_on_v1_path = OUTPUT_DIR / "gles_mip_v2_on_v1_review_template.csv"


In [15]:
v2_review = pd.read_csv(v2_on_v1_path)
v1_review = pd.read_csv(v1_review_path, delimiter=";")

v1_review.shape, v2_review.shape

((95, 38), (95, 38))

In [21]:
fields = [
    "issue_domain",
    "specificity",
    "framing",
    "ambiguity",
    "multi_issue",
]

In [18]:
(v1_review["sample_id"] == v2_review["sample_id"]).all()

np.True_

In [19]:
v1_review["sample_id"].nunique(), v2_review["sample_id"].nunique()

(95, 95)

In [ ]:
rows = []

for field in fields:
    model_col = f"{field}_model"
    reviewed_col = f"{field}_reviewed"
    
    compared = v1_review[reviewed_col].notna()
    
    v1_matches = (
        v1_review.loc[compared, model_col].astype(str)
        == v1_review.loc[compared, reviewed_col].astype(str)
    )
    v2_matches = (
        v2_review.loc[compared, model_col].astype(str)
        == v2_review.loc[compared, reviewed_col].astype(str)
    )
    
    rows.append({
        "field": field,
        "n_compared": compared.sum(),
        "v1_matches": v1_matches.sum(),
        "v1_percent_agreement": v1_matches.mean(),
        "v2_matches": v2_matches.sum(),
        "v2_percent_agreement": v2_matches.mean(),
        "delta": v2_matches.mean() - v1_matches.mean(),
    })
    
agreement_compare = pd.DataFrame(rows)
agreement_compare

,field,n_compared,v1_matches,v1_percent_agreement,v2_matches,v2_percent_agreement,delta
0,issue_domain,95,60,0.631579,71,0.747368,0.115789
1,specificity,95,51,0.536842,61,0.642105,0.105263
2,framing,95,56,0.589474,74,0.778947,0.189474
3,ambiguity,95,40,0.421053,58,0.610526,0.189474
4,multi_issue,95,89,0.936842,85,0.894737,-0.042105


In [ ]:
case_rows = []

for field in fields:
    model_col = f"{field}_model"
    reviewed_col = f"{field}_reviewed"
    
    compared = v1_review[reviewed_col].notna()
    
    v1_match = (
        v1_review.loc[compared, model_col].astype(str)
        == v1_review.loc[compared, reviewed_col].astype(str)
    )
    
    v2_match = (
        v2_review.loc[compared, model_col].astype(str)
        == v2_review.loc[compared, reviewed_col].astype(str)
    )
    
    temp = pd.DataFrame({
        "sample_id": v1_review.loc[compared, "sample_id"],
        "response_text": v1_review.loc[compared, "response_text"],
        "field": field,
        "v1_model": v1_review.loc[compared, model_col],
        "v2_model": v2_review.loc[compared, model_col],
        "reviewed": v1_review.loc[compared, reviewed_col],
        "v1_match": v1_match.values,
        "v2_match": v2_match.values,
    })
    
    temp["change_type"] = "same"
    temp.loc[(~temp["v1_match"]) & (temp["v2_match"]), "change_type"] = "improved"
    temp.loc[(temp["v1_match"]) & (~temp["v2_match"]), "change_type"] = "worsened"
    temp.loc[(~temp["v1_match"]) & (~temp["v2_match"]), "change_type"] = "still_wrong"
    
    case_rows.append(temp)

case_compare = pd.concat(case_rows, ignore_index=True)
case_compare.head()


,sample_id,response_text,field,v1_model,v2_model,reviewed,v1_match,v2_match,change_type
0,gles_mip_v1_0227,Was wird aus der eigenen Bevölkerung,issue_domain,social_welfare,democracy_governance,social_welfare,True,False,worsened
1,gles_mip_v1_0115,Energikosten Sicherheit Frieden und Wirtschaft,issue_domain,security,other,economy,False,False,still_wrong
2,gles_mip_v1_0206,"Die vielen Flüchtlinge, Und die inflation",issue_domain,migration,migration,migration,True,True,same
3,gles_mip_v1_0359,Raus aus dem Ukraine-Krieg. Wir stehen vor ein...,issue_domain,foreign_policy_war,foreign_policy_war,foreign_policy_war,True,True,same
4,gles_mip_v1_0151,Die Uneinigkeit unter einander,issue_domain,democracy_governance,democracy_governance,democracy_governance,True,True,same


In [ ]:
pd.crosstab(case_compare["field"], case_compare["change_type"])

change_type,improved,same,still_wrong,worsened
field,,,,
ambiguity,26,32,29,8
framing,20,54,19,2
issue_domain,14,57,21,3
multi_issue,5,80,1,9
specificity,15,46,29,5


In [36]:
case_compare[case_compare["change_type"] == "improved"].sort_values(["field", "sample_id"]).head(30)

,sample_id,response_text,field,v1_model,v2_model,reviewed,v1_match,v2_match,change_type
347,gles_mip_v1_0011,Immigration und Rente,ambiguity,low,medium,medium,False,True,improved
316,gles_mip_v1_0031,Die Parteien,ambiguity,high,low,low,False,True,improved
313,gles_mip_v1_0039,"Rente, Mindestlohn, Wirtschaft und Grenzkontro...",ambiguity,low,medium,medium,False,True,improved
320,gles_mip_v1_0052,"Migration, wirtschaftliche Lage",ambiguity,low,medium,medium,False,True,improved
305,gles_mip_v1_0062,Jeder ist sich selbst der nächste,ambiguity,high,medium,medium,False,True,improved
377,gles_mip_v1_0109,Tendenz zur AFD in der Bevölkerung,ambiguity,medium,low,low,False,True,improved
339,gles_mip_v1_0114,das es dem eigenen volk besser geht,ambiguity,high,medium,medium,False,True,improved
286,gles_mip_v1_0115,Energikosten Sicherheit Frieden und Wirtschaft,ambiguity,low,medium,medium,False,True,improved
307,gles_mip_v1_0122,Diskriminierung von deutschen,ambiguity,high,medium,medium,False,True,improved
327,gles_mip_v1_0134,"Migration, Energie, Kosten für Miete und Leben...",ambiguity,low,medium,medium,False,True,improved


In [37]:
case_compare[(case_compare["field"] == "framing") & (case_compare["change_type"] == "improved")]


,sample_id,response_text,field,v1_model,v2_model,reviewed,v1_match,v2_match,change_type
190,gles_mip_v1_0227,Was wird aus der eigenen Bevölkerung,framing,descriptive,evaluative,evaluative,False,True,improved
194,gles_mip_v1_0151,Die Uneinigkeit unter einander,framing,descriptive,evaluative,evaluative,False,True,improved
196,gles_mip_v1_0327,Unstimmigkeiten,framing,descriptive,evaluative,evaluative,False,True,improved
202,gles_mip_v1_0158,Die Hetze,framing,descriptive,evaluative,evaluative,False,True,improved
209,gles_mip_v1_0076,Das die sich nicht einig sind,framing,descriptive,evaluative,evaluative,False,True,improved
210,gles_mip_v1_0062,Jeder ist sich selbst der nächste,framing,descriptive,evaluative,evaluative,False,True,improved
216,gles_mip_v1_0070,Uneinigkeit,framing,descriptive,evaluative,evaluative,False,True,improved
222,gles_mip_v1_0343,Unklarheit der Prioritäten: sozial vs. wirtsch...,framing,descriptive,evaluative,evaluative,False,True,improved
224,gles_mip_v1_0222,"Unsichere politische Lage, Energiepreise, Infl...",framing,descriptive,evaluative,evaluative,False,True,improved
231,gles_mip_v1_0061,Die Schaffung eines einheitlichen Kurses,framing,descriptive,directive,directive,False,True,improved


In [43]:
case_compare[case_compare["change_type"] == "worsened"].sort_values(["field", "sample_id"]).head(30)

,sample_id,response_text,field,v1_model,v2_model,reviewed,v1_match,v2_match,change_type
353,gles_mip_v1_0017,Unzufriedenheit allgemein,ambiguity,high,medium,high,True,False,worsened
333,gles_mip_v1_0148,Das nicht an die Bürger des eigenen Landes ged...,ambiguity,medium,low,medium,True,False,worsened
297,gles_mip_v1_0158,Die Hetze,ambiguity,high,low,high,True,False,worsened
338,gles_mip_v1_0201,"Stimmung wird , bei berechtigter Kritik, runte...",ambiguity,high,low,high,True,False,worsened
306,gles_mip_v1_0218,Unser Problem ist die gegenwärtige Regierung b...,ambiguity,low,medium,low,True,False,worsened
285,gles_mip_v1_0227,Was wird aus der eigenen Bevölkerung,ambiguity,high,low,high,True,False,worsened
358,gles_mip_v1_0344,Zuviel Einflußnahme von Aussen (Elon Wlademier...,ambiguity,medium,low,medium,True,False,worsened
310,gles_mip_v1_0373,"Inkompetenz der Poliker was die Themen Wohnen,...",ambiguity,medium,low,medium,True,False,worsened
200,gles_mip_v1_0174,"Flüchtlinge, Steuern, erhöhte Kosten, niedrige...",framing,descriptive,evaluative,descriptive,True,False,worsened
192,gles_mip_v1_0206,"Die vielen Flüchtlinge, Und die inflation",framing,descriptive,evaluative,descriptive,True,False,worsened


In [44]:
case_compare[(case_compare["field"] == "ambiguity") & (case_compare["change_type"] == "worsened")]

,sample_id,response_text,field,v1_model,v2_model,reviewed,v1_match,v2_match,change_type
285,gles_mip_v1_0227,Was wird aus der eigenen Bevölkerung,ambiguity,high,low,high,True,False,worsened
297,gles_mip_v1_0158,Die Hetze,ambiguity,high,low,high,True,False,worsened
306,gles_mip_v1_0218,Unser Problem ist die gegenwärtige Regierung b...,ambiguity,low,medium,low,True,False,worsened
310,gles_mip_v1_0373,"Inkompetenz der Poliker was die Themen Wohnen,...",ambiguity,medium,low,medium,True,False,worsened
333,gles_mip_v1_0148,Das nicht an die Bürger des eigenen Landes ged...,ambiguity,medium,low,medium,True,False,worsened
338,gles_mip_v1_0201,"Stimmung wird , bei berechtigter Kritik, runte...",ambiguity,high,low,high,True,False,worsened
353,gles_mip_v1_0017,Unzufriedenheit allgemein,ambiguity,high,medium,high,True,False,worsened
358,gles_mip_v1_0344,Zuviel Einflußnahme von Aussen (Elon Wlademier...,ambiguity,medium,low,medium,True,False,worsened


## Initial findings

Using the same 95 human-reviewed rows, the v2 codebook improves agreement with the reviewed labels on four of the five coded variables. The largest gains are in `framing` and `ambiguity`, followed by meaningful gains in `issue_domain` and `specificity`. This suggests that the v2 additions — especially `coding_procedure` and `tie_breakers` — changed the model behavior in the intended direction rather than only making the codebook more verbose.

The clearest success is `framing`. Many improved cases are short or nominalized German complaints such as *Uneinigkeit*, *Unstimmigkeiten*, *Die Hetze*, or *Keine klare Linie der Ziele*. These were often coded by v1 as `descriptive`, while v2 more often matches the reviewed `evaluative` or `directive` labels.

The main regression is `multi_issue`: v2 loses a small amount of agreement compared with v1. Since v1 was already very strong on this variable, this may indicate over-tightening of the v2 rule that requires distinct issue domains.

`ambiguity` improves substantially overall, but the worsened cases suggest a possible overcorrection. Some responses reviewed as `high` or `medium` ambiguity are now coded by v2 as `low`, meaning that v2 may be too confident for some short abstract or interpretively loaded responses.

### Next checks

1. Inspect `multi_issue` worsened cases to see whether v2 is under-detecting genuinely multi-domain responses.
2. Inspect `ambiguity` worsened cases to see whether v2 is overusing `low`.
3. Use these patterns to decide whether a small v3 codebook revision is needed, especially for ambiguity calibration and multi-issue boundary rules.
